# Detector de Caracteres — Treino (YOLO)

Notebook principal de treino do detector agnóstico de caracteres (classe única `character`), etapa 1 do pipeline detector → classificador N1 → filtro N1.

Dataset de anotações: `miguelmussalam/manga109-character-bouding-box` (Kaggle) — páginas do Manga109 com bboxes em nível de caractere, anotadas via Roboflow.

**Checklist antes de rodar:**
1. Anexe o dataset de anotações no painel lateral direito em **+ Add Input → Datasets**.
2. Habilite a GPU: *Session options → Accelerator → GPU T4 x2 ou P100*.
3. **Run All**.

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "GPU nao encontrada!")

## 1. Parâmetros do experimento

**Altere apenas esta célula** para controlar o treino inteiro. As variáveis `KD_*` são lidas por `config.py` via `os.environ` — sem elas definidas, `config.py` usa seus próprios defaults.

In [ ]:
import os

# --- Modelo e treino YOLO ---
YOLO_MODEL      = "yolo26n.pt"   # ou "yolo11n.pt", "yolo11s.pt", etc.
EPOCHS          = 150
IMGSZ           = 1024           # caracteres pequenos: 1024 preserva mais detalhe que 640
BATCH           = 8              # ajustar para caber na VRAM (T4/P100 = 16GB); OOM -> reduzir

# --- Workers do dataloader (paralelismo de I/O) ---
KAGGLE_WORKERS  = 2              # T4/P100 do Kaggle tem 2 vCPUs
LOCAL_WORKERS   = 4

# --- Organizacao dos runs (pasta de resultados) ---
PROJECT_NAME    = "kanji_detector"

# ============================================================
# Aplica as configuracoes como variaveis de ambiente
# (KD_DATA_YAML e setado automaticamente na Celula 10, depois do split)
# ============================================================
os.environ["KD_YOLO_MODEL"]     = str(YOLO_MODEL)
os.environ["KD_EPOCHS"]         = str(EPOCHS)
os.environ["KD_IMGSZ"]          = str(IMGSZ)
os.environ["KD_BATCH"]          = str(BATCH)
os.environ["KD_KAGGLE_WORKERS"] = str(KAGGLE_WORKERS)
os.environ["KD_LOCAL_WORKERS"]  = str(LOCAL_WORKERS)
os.environ["KD_PROJECT_NAME"]   = str(PROJECT_NAME)

print("Parametros registrados!")
print(f"  Modelo: {YOLO_MODEL} | Epochs: {EPOCHS} | Imgsz: {IMGSZ} | Batch: {BATCH}")

## 2. Instalar dependências

In [ ]:
!pip install -q ultralytics
print("Dependencias instaladas.")

## 3. Configurar repositório

Clona (ou atualiza, se já clonado nesta sessão) o repositório com o código do pipeline (`config.py`, `src/detector/train.py`) em `/kaggle/working/` e adiciona ao `sys.path` para permitir `from src.detector...`.

In [ ]:
import os
import sys
import shutil

WORK_DIR  = "/kaggle/working"
REPO_NAME = "Detector-de-kanjis-n1"
REPO_DIR  = os.path.join(WORK_DIR, REPO_NAME)
REPO_URL  = f"https://github.com/MiguelMussalam/{REPO_NAME}.git"

is_valid_repo = os.path.isdir(os.path.join(REPO_DIR, ".git"))

if not is_valid_repo:
    if os.path.exists(REPO_DIR):
        print(f"Diretorio {REPO_DIR} existe mas nao e um repo git valido. Removendo...")
        shutil.rmtree(REPO_DIR)
    print(f"Clonando {REPO_URL} ...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo valido encontrado. Atualizando...")
    !git -C {REPO_DIR} pull

assert os.path.isfile(os.path.join(REPO_DIR, "config.py")), \
    f"config.py nao encontrado em {REPO_DIR} — verifique se o clone funcionou"

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from config import PROJECT_NAME

print(f"Diretorio de trabalho: {os.getcwd()}")
print(f"Project name (runs):   {PROJECT_NAME}")

## 4. Preparar dataset

O dataset anotado (Kaggle Input) é read-only. Este passo localiza a pasta com `images/` e `labels/` dentro de `/kaggle/input/` (a estrutura de nesting varia conforme o ambiente Kaggle), copia para `/kaggle/working/` e cria um split determinístico train/val (85/15, seed fixa).

In [ ]:
import random
from pathlib import Path

def encontrar_dataset(root="/kaggle/input"):
    """Busca recursiva por uma pasta contendo subpastas images/ e labels/."""
    candidatos = []
    for dirpath, dirnames, _ in os.walk(root):
        if "images" in dirnames and "labels" in dirnames:
            candidatos.append(Path(dirpath))
    return candidatos

candidatos = encontrar_dataset()
print(f"Candidatos encontrados: {len(candidatos)}")
for c in candidatos:
    print(" ", c)

if not candidatos:
    raise FileNotFoundError(
        "Nenhuma pasta com 'images' e 'labels' encontrada em /kaggle/input. "
        "Verifique se o dataset foi anexado ao notebook."
    )

SRC = candidatos[0]
SRC_IMG = SRC / "images"
SRC_LBL = SRC / "labels"
print(f"\nUsando: {SRC}")

WORK = Path("/kaggle/working/manga_char")

imgs = sorted([p for p in SRC_IMG.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
print(f"Total de imagens: {len(imgs)}")

random.seed(42)
random.shuffle(imgs)

val_count = max(1, round(len(imgs) * 0.15))
val_imgs = set(imgs[:val_count])
train_imgs = set(imgs[val_count:])

for split, group in [("train", train_imgs), ("val", val_imgs)]:
    (WORK / split / "images").mkdir(parents=True, exist_ok=True)
    (WORK / split / "labels").mkdir(parents=True, exist_ok=True)
    for img_path in group:
        lbl_path = SRC_LBL / (img_path.stem + ".txt")
        shutil.copy(img_path, WORK / split / "images" / img_path.name)
        if lbl_path.exists():
            shutil.copy(lbl_path, WORK / split / "labels" / lbl_path.name)
        else:
            print(f"AVISO: label ausente para {img_path.name}")

print(f"Treino: {len(list((WORK/'train/images').iterdir()))} imgs")
print(f"Val:    {len(list((WORK/'val/images').iterdir()))} imgs")

## 5. Gerar `data.yaml`

In [ ]:
import yaml

data_yaml = {
    "path": str(WORK),
    "train": "train/images",
    "val": "val/images",
    "nc": 1,
    "names": ["character"],
}

yaml_path = WORK / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

os.environ["KD_DATA_YAML"] = str(yaml_path)

print(yaml_path.read_text())
print(f"KD_DATA_YAML setado para: {yaml_path}")

## 6. Treinar

Chama `src.detector.train`, que lê `DATA_YAML` e os hiperparâmetros (`YOLO_MODEL`, `EPOCHS`, `IMGSZ`, `BATCH`, workers) do `config.py`. Para ajustar algum hiperparâmetro nesta sessão, defina a env var correspondente (`KD_EPOCHS`, `KD_BATCH`, etc.) antes de rodar a célula abaixo.

In [ ]:
!python -m src.detector.train

## 7. Resultados e curvas

In [ ]:
import pandas as pd
from IPython.display import Image, display

RUNS_DIR = Path(REPO_DIR) / PROJECT_NAME / "run"
results_csv = RUNS_DIR / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]
    print(f"mAP@50:      {last.get('metrics/mAP50(B)', float('nan')):.4f}")
    print(f"mAP@50-95:   {last.get('metrics/mAP50-95(B)', float('nan')):.4f}")
    print(f"Precision:   {last.get('metrics/precision(B)', float('nan')):.4f}")
    print(f"Recall:      {last.get('metrics/recall(B)', float('nan')):.4f}")
else:
    print(f"results.csv nao encontrado em {results_csv}")

for fname, titulo in [
    ("results.png",          "=== Curvas de Treino (mAP / Loss) ==="),
    ("confusion_matrix.png", "=== Matriz de Confusao ==="),
    ("val_batch0_pred.jpg",  "=== Predicoes no conjunto de validacao ==="),
]:
    fpath = RUNS_DIR / fname
    if fpath.exists():
        print(titulo)
        display(Image(str(fpath)))

## 8. Inspeção visual em página fora do treino

Roda o `best.pt` numa página de manga não usada no treino/validação — ajuste `pagina_teste` abaixo. Parâmetros de inferência calibrados na rodada anterior: `conf=0.30`, `iou=0.40`, `max_det=1000` (páginas de manga têm 200+ caracteres, acima do default de 300 detecções).

In [ ]:
from ultralytics import YOLO

best = YOLO(str(RUNS_DIR / "weights" / "best.pt"))

# Ajuste para o caminho de uma pagina de manga fora do dataset de treino/validacao
pagina_teste = "/kaggle/input/CAMINHO/PARA/pagina_teste.jpg"

if os.path.exists(pagina_teste):
    results = best.predict(
        pagina_teste,
        imgsz=1024,
        conf=0.30,       # sobe um pouco para reduzir bboxes marginais
        iou=0.40,        # NMS mais agressivo para eliminar duplicatas
        agnostic_nms=True,
        max_det=1000,    # default e 300, paginas tem 200+ caracteres
        save=True,
        project="/kaggle/working/predict",
        name="fora_do_treino",
    )
    for r in results:
        print(f"{Path(r.path).name}: {len(r.boxes)} deteccoes")
else:
    print(f"Ajuste 'pagina_teste' para uma imagem valida. Caminho atual nao existe: {pagina_teste}")

## 9. Compactar e baixar

In [ ]:
import zipfile
from IPython.display import FileLink, display

zip_name = "/kaggle/working/resultados_detector.zip"

def zipdir(path, ziph, arcbase):
    for root, dirs, files in os.walk(path):
        for file in files:
            filepath = os.path.join(root, file)
            arcname  = os.path.join(arcbase, os.path.relpath(filepath, path))
            ziph.write(filepath, arcname)

print(f"Criando {zip_name}...")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    if RUNS_DIR.exists():
        zipdir(str(RUNS_DIR), zipf, "run")
        print("Resultados YOLO adicionados.")
    else:
        print(f"AVISO: pasta de runs nao encontrada: {RUNS_DIR}")

    config_file = os.path.join(REPO_DIR, "config.py")
    if os.path.exists(config_file):
        zipf.write(config_file, "config.py")
        print("config.py adicionado.")

print(f"\nZip criado: {zip_name}")
display(FileLink(zip_name))